# 04 · Optimización y escalabilidad en EMR

Ejecutar en el nodo primario de EMR con el kernel **American Tower - PySpark**, en un kernel nuevo y en orden. Cada celda muestra la evidencia que produce. Los registros de Spark y el resultado quedan enlazados por el identificador de corrida.

Evalúa la unión de teselas Silver con el contexto distrital Gold usando shuffle y broadcast. Luego reproduce las
transformaciones Bronze a Gold a 1×, 10× y 100× en rutas de prueba dentro de Operations. Las réplicas afectan a
las teselas y conservan las mismas dimensiones. La evidencia registra los recursos solicitados y los workers con
tareas cuando la interfaz Spark ofrece esos datos.

## 1. Configuración de la prueba

Las mediciones comparan las estrategias en el mismo SparkContext. Mantener los mismos recursos durante la corrida.

In [ ]:
import pandas as pd
from IPython.display import display


def show_values(values):
    show_rows(values.items(), ["Indicador", "Valor"])


def show_rows(rows, columns=None):
    with pd.option_context("display.max_colwidth", None):
        display(pd.DataFrame(rows, columns=columns))

REGION = "us-east-1"
OPERATIONS_BUCKET = "american-tower-dev-operations-1dd72256126674a5893dfca3df"
GOLD_RUN_URI = "s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073334Z-7ea4f607ff/gold/_SUCCESS.json"  # Pegar la manifest_uri de entrada.
EXECUTORS = 10
EXECUTOR_CORES = 4
EXECUTOR_MEMORY = "8g"
EXECUTOR_OVERHEAD = "3g"
SHUFFLE_PARTITIONS = EXECUTORS * 12

BENCHMARK_PERIOD = None  # Ejemplo: "2024-01-01" para un solo trimestre.
SCALE_FACTORS = (1, 10, 100)
TEMP_DIR = "/mnt"

show_values({"Región": REGION, "Manifiesto de entrada": GOLD_RUN_URI or "Pendiente de configurar", "Bucket Operations": OPERATIONS_BUCKET, "Executors solicitados": EXECUTORS})
if not SCALE_FACTORS or any(factor not in (1, 10, 100) for factor in SCALE_FACTORS):
    raise ValueError("Elegir uno o más factores entre 1, 10 y 100.")
show_values({"Lote": BENCHMARK_PERIOD or "Todos los trimestres de Bronze",
             "Factores": ", ".join(f"{x}×" for x in sorted(set(SCALE_FACTORS)))})

,Indicador,Valor
0,Región,us-east-1
1,Manifiesto de entrada,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073334Z-7ea4f607ff/gold/_SUCCESS.json
2,Bucket Operations,american-tower-dev-operations-1dd72256126674a5893dfca3df
3,Executors solicitados,8


,Indicador,Valor
0,Lote,Todos los trimestres de Bronze
1,Factores,"1×, 10×, 100×"


## 2. Linaje y entradas de las pruebas

In [2]:
import json
import uuid
from datetime import datetime, timezone

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

s3 = boto3.client("s3", region_name=REGION, config=Config(retries={"mode": "standard", "max_attempts": 5}))
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:10]


def s3_path(uri):
    if not isinstance(uri, str) or not uri.startswith("s3://"):
        raise ValueError(f"Se requiere una URI S3 completa: {uri!r}")
    bucket, slash, key = uri.removeprefix("s3://").partition("/")
    if not bucket or not slash or not key:
        raise ValueError(f"Se requiere una URI S3 completa: {uri}")
    return bucket, key


def missing_object(error):
    return error.response["Error"]["Code"] in {"NoSuchKey", "NotFound", "404"}


def read_json(uri):
    bucket, key = s3_path(uri)
    try:
        response = s3.get_object(Bucket=bucket, Key=key)
    except ClientError as error:
        if missing_object(error):
            raise FileNotFoundError(f"No existe el manifiesto {uri}") from error
        raise
    with response["Body"] as stream:
        return json.load(stream)


def save_json(uri, value):
    bucket, key = s3_path(uri)
    s3.put_object(
        Bucket=bucket, Key=key, IfNoneMatch="*", ContentType="application/json",
        Body=json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False).encode("utf-8"),
    )
    return uri



def read_manifest(uri, schema, parameter):
    if not uri:
        raise ValueError(f"Completar {parameter} con la manifest_uri que devolvió la etapa anterior.")
    document = read_json(uri)
    if document.get("schema") != schema or document.get("complete") is not True or not document.get("run_id"):
        raise ValueError(f"Manifiesto incompleto o de un tipo diferente a {schema}: {uri}")
    return document

from pathlib import PurePosixPath
import re

PERIOD_FILE = re.compile(r"^(\d{4})-(01|04|07|10)-01_performance_mobile_tiles\.parquet$")


def object_period(item):
    match = PERIOD_FILE.fullmatch(PurePosixPath(item["key"]).name)
    if not match:
        raise ValueError(f"Nombre de Parquet trimestral inesperado: {item['key']}")
    year, month = map(int, match.groups())
    return year * 4 + (month - 1) // 3


def validate_inventory(items):
    mobile = [item for item in items if item["source"] == "ookla" and item["key"].endswith(".parquet")]
    periods = [object_period(item) for item in mobile]
    if not periods or len(periods) != len(set(periods)):
        raise ValueError("Se requiere un solo Parquet Ookla móvil por trimestre.")
    coverage = [item for item in items if item["source"] == "osiptel_cobertura"
                and item["key"].lower().endswith(".csv")]
    districts = [item for item in items if item["source"] == "inei" and
                 re.fullmatch(r"distritos?\.(rar|zip|gpkg)", PurePosixPath(item["key"]).name.lower())]
    worldpop = [item for item in items if item["source"] == "worldpop"
                and PurePosixPath(item["key"]).name.lower() == "per_ppp_2020.tif"]
    if not all(len(group) == 1 for group in (coverage, districts, worldpop)):
        raise ValueError("El inventario requiere un CSV OSIPTEL, la capa distrital INEI y WorldPop 2020.")
    return mobile, coverage[0], districts[0], worldpop[0]


def verified_uri(item):
    try:
        header = s3.head_object(Bucket=item["bucket"], Key=item["key"])
    except ClientError as error:
        if missing_object(error):
            raise FileNotFoundError(f"No existe el objeto publicado en Bronze: {item['key']}") from error
        raise
    if (item.get("status") not in {"verified", "reused_verified"}
            or header["ContentLength"] != item["bytes"]
            or header.get("VersionId") != item.get("version_id")
            or header["Metadata"].get("sha256") != item["sha256"]):
        raise ValueError(f"El objeto difiere de su recibo Bronze: {item['key']}")
    return f"s3://{item['bucket']}/{item['key']}"
gold = read_manifest(GOLD_RUN_URI, "gold-autonomo-v2", "GOLD_RUN_URI")
silver_uri = gold["silver_manifest_uri"]
silver = read_manifest(silver_uri, "silver-autonomo-v1", "silver_manifest_uri")
bronze_uri = silver["bronze_manifest_uri"]
bronze = read_manifest(bronze_uri, "bronze-autonomo-v1", "bronze_manifest_uri")
if (silver["run_id"] != gold["silver_run_id"] or bronze["run_id"] != silver["bronze_run_id"]
        or bronze_uri != gold["bronze_manifest_uri"]):
    raise ValueError("El linaje Bronze, Silver y Gold no concilia.")
lineage = {"gold_manifest_uri": GOLD_RUN_URI, "silver_manifest_uri": silver_uri, "bronze_manifest_uri": bronze_uri}
root = f"s3://{OPERATIONS_BUCKET}/run_id={RUN_ID}/rendimiento"
show_values({**lineage, "Teselas reportadas por Silver": silver["quality"]["espacial"]["asignadas"],
             "Distritos de contexto Gold": gold["context"]["filas"], "Salida aislada": root})

/opt/american-tower/venv/lib64/python3.9/site-packages/boto3/compat.py:89: PythonDeprecationWarning: Boto3 will no longer support Python 3.9 starting April 29, 2026. To continue receiving service updates, bug fixes, and security updates please upgrade to Python 3.10 or later. More information can be found here: https://aws.amazon.com/blogs/developer/python-support-policy-updates-for-aws-sdks-and-tools/
  warnings.warn(warning, PythonDeprecationWarning)


,Indicador,Valor
0,gold_manifest_uri,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073334Z-7ea4f607ff/gold/_SUCCESS.json
1,silver_manifest_uri,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T072542Z-e92a810559/silver/_SUCCESS.json
2,bronze_manifest_uri,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T071603Z-190d23d89e/bronze/_SUCCESS.json
3,Teselas reportadas por Silver,417420
4,Distritos de contexto Gold,1890
5,Salida aislada,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento


## 3. Inicialización de Spark en YARN

In [3]:
import os
import sys
from contextlib import contextmanager
from pathlib import Path

SPARK_HOME = Path("/usr/lib/spark")
py4j = sorted((SPARK_HOME / "python/lib").glob("py4j-*.zip"))
if not py4j:
    raise RuntimeError("Abrir esta notebook en el nodo primario de Amazon EMR con Spark instalado.")
os.environ["SPARK_HOME"] = str(SPARK_HOME)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
sys.path[:0] = [str(SPARK_HOME / "python"), str(py4j[0])]

from pyspark import SparkContext, StorageLevel
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import StructType


@contextmanager
def spark_session(stage):
    if SparkContext._active_spark_context is not None:
        raise RuntimeError("Reiniciar el kernel para que YARN aplique los parámetros de esta corrida.")
    # EMRFS necesita un prefijo existente para el registro de eventos.
    log_dir = f"run_id={RUN_ID}/eventos_spark/"
    s3.put_object(Bucket=OPERATIONS_BUCKET, Key=log_dir, Body=b"")
    spark = (
        SparkSession.builder.master("yarn").appName(f"american-tower-{stage}")
        .config("spark.submit.deployMode", "client")
        .config("spark.pyspark.python", sys.executable)
        .config("spark.executorEnv.PYSPARK_PYTHON", sys.executable)
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.executor.instances", str(EXECUTORS))
        .config("spark.executor.cores", str(EXECUTOR_CORES))
        .config("spark.executor.memory", EXECUTOR_MEMORY)
        .config("spark.executor.memoryOverhead", EXECUTOR_OVERHEAD)
        .config("spark.sql.shuffle.partitions", str(SHUFFLE_PARTITIONS))
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.eventLog.enabled", "true")
        .config("spark.eventLog.dir", f"s3://{OPERATIONS_BUCKET}/{log_dir}")
        .getOrCreate()
    )
    try:
        spark.sparkContext.setLogLevel("WARN")
        print(f"Aplicación Spark {spark.sparkContext.applicationId}", flush=True)
        yield spark
    finally:
        spark.stop()


def write_table(frame, uri, partitions=()):
    writer = frame.write.mode("errorifexists").option("compression", "snappy")
    if partitions:
        writer = writer.partitionBy(*partitions)
    writer.parquet(uri)
    return uri

from contextlib import ExitStack
from IPython import get_ipython
from urllib.parse import quote
from urllib.request import urlopen


def executor_usage(sc):
    if not sc.uiWebUrl:
        return {"estado": "interfaz Spark deshabilitada", "activos": None, "tareas": {}, "hosts": {}}
    app = quote(sc.applicationId, safe="")
    try:
        with urlopen(f"{sc.uiWebUrl.rstrip('/')}/api/v1/applications/{app}/allexecutors", timeout=4) as response:
            rows = [row for row in json.load(response) if row["id"] != "driver"]
    except (OSError, ValueError, KeyError) as error:
        return {"estado": f"interfaz no disponible ({type(error).__name__})",
                "activos": None, "tareas": {}, "hosts": {}}
    return {"estado": "lectura disponible", "activos": sum(row.get("isActive", False) for row in rows),
            "tareas": {row["id"]: row.get("totalTasks", 0) for row in rows},
            "hosts": {row["id"]: row.get("hostPort", "").rsplit(":", 1)[0] for row in rows}}


def show_spark(spark, events_uri):
    sc = spark.sparkContext
    show_values({"Aplicación": sc.applicationId, "Spark": spark.version, "Master": sc.master,
                 "Paralelismo base": sc.defaultParallelism, "Executors solicitados": EXECUTORS,
                 "vCores por executor": EXECUTOR_CORES,
                 "Heap y overhead por executor": f"{EXECUTOR_MEMORY} + {EXECUTOR_OVERHEAD}",
                 "Particiones shuffle": spark.conf.get("spark.sql.shuffle.partitions"),
                 "Eventos Spark": events_uri})


def open_stage(stage):
    events = get_ipython().events
    resources = ExitStack()

    def stop_on_failure(result):
        if result.error_before_exec or result.error_in_exec:
            resources.close()

    events.register("post_run_cell", stop_on_failure)
    resources.callback(events.unregister, "post_run_cell", stop_on_failure)
    try:
        spark = resources.enter_context(spark_session(stage))
    except BaseException:
        resources.close()
        raise
    events_uri = f"s3://{OPERATIONS_BUCKET}/run_id={RUN_ID}/eventos_spark/"
    return spark, resources, events_uri

spark, resources, events_uri = open_stage("rendimiento")
show_spark(spark, events_uri)

Nov 25, 2026 7:35:52 AM org.apache.spark.launcher.Log4jHotPatchOption staticJavaAgentOption

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/25 07:35:56 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Aplicación Spark application_1795587001536_0006


,Indicador,Valor
0,Aplicación,application_1795587001536_0006
1,Spark,3.5.0-amzn-1
2,Master,yarn
3,Paralelismo base,2
4,Executors solicitados,8
5,vCores por executor,3
6,Heap y overhead por executor,8g + 3g
7,Particiones shuffle,96
8,Eventos Spark,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/eventos_spark/


## 4. Comparación de unión shuffle y broadcast

Se verifica la igualdad de filas entre ambas estrategias antes de presentar sus tiempos. Un porcentaje negativo indica que broadcast demoró más en la medición.

In [4]:
import time
from statistics import median


def benchmark_join(spark, silver, gold):
    fact = spark.read.parquet(silver["tables"]["ookla_teselas"]).select("quadkey", "period_id", "ubigeo").cache()
    dim = spark.read.parquet(gold["context_uri"]).select("ubigeo", "departamento").cache()
    settings = {"spark.sql.autoBroadcastJoinThreshold": "-1", "spark.sql.adaptive.enabled": "false"}
    previous = {key: spark.conf.get(key) for key in settings}

    def measure(frame):
        start = time.perf_counter()
        row = frame.agg(
            F.count("*").alias("n"),
            F.expr("sum(CAST(xxhash64(quadkey, period_id, ubigeo, departamento) AS DECIMAL(38,0)))").alias("hash"),
        ).first()
        return {"seconds": time.perf_counter() - start, "rows": row.n, "hash": str(row.hash)}

    try:
        expected = fact.count()
        if not expected or dim.count() != dim.select("ubigeo").distinct().count():
            raise ValueError("Se requieren teselas reales y una dimensión UBIGEO sin duplicados.")
        for key, value in settings.items():
            spark.conf.set(key, value)
        joins = {"shuffle": fact.hint("MERGE").join(dim.hint("MERGE"), "ubigeo"),
                 "broadcast": fact.join(F.broadcast(dim), "ubigeo")}
        plans = {name: frame._jdf.queryExecution().executedPlan().toString() for name, frame in joins.items()}
        if "SortMergeJoin" not in plans["shuffle"] or "BroadcastHashJoin" not in plans["broadcast"]:
            raise ValueError("Los planes físicos no corresponden a las estrategias que se están comparando.")
        warmup = [measure(frame) for frame in joins.values()]
        if any(item["rows"] != expected for item in warmup):
            raise ValueError("La unión debe conservar todas las teselas asignadas.")
        left, right = joins["shuffle"], joins["broadcast"]
        if left.exceptAll(right).take(1) or right.exceptAll(left).take(1):
            raise ValueError("Las dos estrategias produjeron filas diferentes.")

        timings = {name: [] for name in joins}
        for order in (("shuffle", "broadcast"), ("broadcast", "shuffle"), ("shuffle", "broadcast")):
            for name in order:
                timings[name].append(measure(joins[name]))
        if len({(item["rows"], item["hash"]) for trial in timings.values() for item in trial}) != 1:
            raise ValueError("Las mediciones no son equivalentes.")
        values = {name: median(t["seconds"] for t in trials) for name, trials in timings.items()}
        return {"expected_rows": expected, "trials": timings, "median_seconds": values, "plans": plans,
                "reduction_percent": (values["shuffle"] - values["broadcast"]) / values["shuffle"] * 100}
    finally:
        for key, value in previous.items():
            spark.conf.set(key, value)
        fact.unpersist()
        dim.unpersist()

join = benchmark_join(spark, silver, gold)
join_uri = save_json(f"{root}/union.json", {**lineage, **join, "application_id": spark.sparkContext.applicationId})
join_rows = [{"estrategia": name, "repeticion": index, "segundos": value["seconds"], "filas": value["rows"]}
             for name, observations in join["trials"].items()
             for index, value in enumerate(observations, 1)]
show_rows(join_rows)
show_values({"Filas equivalentes": join["expected_rows"],
             "Mediana shuffle (s)": round(join["median_seconds"]["shuffle"], 3),
             "Mediana broadcast (s)": round(join["median_seconds"]["broadcast"], 3),
             "Variación medida (%)": round(join["reduction_percent"], 2), "Evidencia y planes": join_uri})
for name, plan in join["plans"].items():
    operators = [line.strip() for line in plan.splitlines() if "SortMergeJoin" in line or "BroadcastHashJoin" in line]
    print(f"{name}: {' | '.join(operators)}")

,estrategia,repeticion,segundos,filas
0,shuffle,1,0.971658,417420
1,shuffle,2,0.837797,417420
2,shuffle,3,0.726790,417420
3,broadcast,1,0.429267,417420
4,broadcast,2,0.381404,417420
5,broadcast,3,0.354244,417420


,Indicador,Valor
0,Filas equivalentes,417420
1,Mediana shuffle (s),0.838
2,Mediana broadcast (s),0.381
3,Variación medida (%),54.48
4,Evidencia y planes,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/union.json


shuffle: +- *(5) SortMergeJoin [ubigeo#6], [ubigeo#35], Inner
broadcast: +- *(2) BroadcastHashJoin [ubigeo#6], [ubigeo#35], Inner, BuildRight, false


## 5. Preparación del lote para las pruebas de escala

La descarga de fuentes de contexto al disco temporal ocurre antes de medir. Las lecturas Spark y escrituras Parquet sí se cronometran.

In [5]:
import hashlib
import math
import subprocess
from tempfile import TemporaryDirectory

import geopandas as gpd
import numpy as np
import rasterio
import shapely
from rasterio.mask import mask


def downloaded_object(item, folder):
    uri = verified_uri(item)
    bucket, key = s3_path(uri)
    path = folder / PurePosixPath(key).name
    extra = {"VersionId": item["version_id"]} if item.get("version_id") else None
    s3.download_file(bucket, key, str(path), ExtraArgs=extra)
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024**2), b""):
            digest.update(chunk)
    if digest.hexdigest() != item["sha256"] or path.stat().st_size != item["bytes"]:
        raise ValueError(f"El original descargado no coincide con su recibo: {uri}")
    return path


def load_districts(path):
    if path.suffix.lower() in {".rar", ".zip"}:
        names = subprocess.check_output(["bsdtar", "-tf", str(path)], text=True).splitlines()
        geopackages = [name for name in names if name.lower().endswith(".gpkg")]
        if len(geopackages) != 1:
            raise ValueError("El archivo distrital debe contener un solo GeoPackage.")
        output = path.with_suffix(".gpkg")
        with output.open("wb") as stream:
            subprocess.run(["bsdtar", "-xOf", str(path), geopackages[0]], stdout=stream, check=True)
        path = output
    data = gpd.read_file(path)
    data.columns = data.columns.str.strip().str.lower()
    data = data[["ubigeo", "nombdep", "nombprov", "nombdist", "geometry"]].to_crs(4326)
    data["ubigeo"] = data.ubigeo.astype("string").str.strip().str.zfill(6)
    if not data.ubigeo.str.fullmatch(r"\d{6}").fillna(False).all() or data.ubigeo.duplicated().any():
        raise ValueError("La cartografía debe contener UBIGEO distritales únicos de seis dígitos.")
    if data.geometry.isna().any() or data.geometry.is_empty.any() or data.empty:
        raise ValueError("La cartografía contiene geometrías ausentes.")
    repair = ~data.geometry.is_valid
    data.loc[repair, "geometry"] = data.loc[repair].geometry.buffer(0)
    if (not data.geometry.is_valid.all() or data.geometry.is_empty.any()
            or not data.geometry.geom_type.isin(["Polygon", "MultiPolygon"]).all()):
        raise ValueError("La cartografía no pudo normalizarse a polígonos válidos.")
    data["superficie_km2"] = data.to_crs(6933).area / 1_000_000
    return data.sort_values("ubigeo").reset_index(drop=True)


def district_frame(spark, data):
    rows = [(r.ubigeo, r.nombdep, r.nombprov, r.nombdist, r.superficie_km2) for r in data.itertuples()]
    return spark.createDataFrame(rows, "ubigeo string, departamento string, provincia string, "
                                       "distrito string, superficie_km2 double")


def assign_districts(frame, geo):
    schema = StructType(list(frame.schema.fields)).add("ubigeo", "string").add("coincidencias", "integer")

    def locate(batches):
        codes, polygons = geo.value
        tree = shapely.STRtree(shapely.from_wkb(polygons))
        for batch in batches:
            points = shapely.points(batch.tile_x.to_numpy(), batch.tile_y.to_numpy())
            point_index, polygon_index = tree.query(points, predicate="covered_by")
            matches = np.bincount(point_index, minlength=len(batch)).astype("int32")
            unique = matches[point_index] == 1
            district = np.full(len(batch), None, dtype=object)
            district[point_index[unique]] = codes[polygon_index[unique]]
            yield batch.assign(ubigeo=district, coincidencias=matches)

    return frame.mapInPandas(locate, schema)


def population_frame(spark, districts, path):
    rows = []
    with rasterio.open(path) as dataset:
        if dataset.count != 1 or dataset.crs is None or dataset.nodata is None:
            raise ValueError("WorldPop requiere una banda con CRS y valor NoData.")
        for district in districts.to_crs(dataset.crs).itertuples():
            try:
                pixels, _ = mask(dataset, [district.geometry], crop=True, filled=False)
                values = pixels.compressed()
                values = values[np.isfinite(values) & (values >= 0)]
            except ValueError as error:
                if "do not overlap" not in str(error):
                    raise
                values = np.array([])
            estimate = float(np.sum(values, dtype=np.float64)) if values.size else None
            rows.append((district.ubigeo, estimate, int(values.size)))
    return spark.createDataFrame(rows, "ubigeo string, poblacion_2020 double, pixeles_validos long")

def read_mobile(spark, paths, bounds):
    raw = spark.read.parquet(*paths)
    pattern = r"/(\d{4})-(01|04|07|10)-01_performance_mobile_tiles\.parquet$"
    year = F.regexp_extract(F.input_file_name(), pattern, 1).cast("int")
    month = F.regexp_extract(F.input_file_name(), pattern, 2).cast("int")
    tiles = raw.select("quadkey", F.col("avg_d_kbps").cast("long").alias("avg_d_kbps"),
                       F.col("tests").cast("long").alias("tests"),
                       F.col("devices").cast("long").alias("devices"),
                       (year * 4 + ((month - 1) / 3).cast("int")).alias("period_id"))

    # La tesela de zoom 6 filtra el área amplia, incluso en archivos sin tile_x ni tile_y.
    west, south, east, north = map(float, bounds)
    x0, x1 = [int((lon + 180) * 64 / 360) for lon in (west, east)]
    y0, y1 = [int((1 - math.asinh(math.tan(math.radians(lat))) / math.pi) * 32) for lat in (north, south)]
    prefixes = ["".join(str(((x >> bit) & 1) + 2 * ((y >> bit) & 1)) for bit in range(5, -1, -1))
                for x in range(x0, x1 + 1) for y in range(y0, y1 + 1)]
    candidates = tiles.filter(F.substring("quadkey", 1, 6).isin(prefixes) & F.col("quadkey").rlike(r"^[0-3]{16}$"))
    x = F.expr("aggregate(sequence(1, 16), 0, (a, i) -> a * 2 + (int(substring(quadkey, i, 1)) & 1))")
    y = F.expr("aggregate(sequence(1, 16), 0, (a, i) -> a * 2 + int(int(substring(quadkey, i, 1)) DIV 2))")
    lat = F.degrees(F.atan(F.sinh(math.pi * (1 - 2 * (y + 0.5) / 65536))))
    return (candidates.withColumn("tile_x", (x + 0.5) / 65536 * 360 - 180)
            .withColumn("tile_y", lat)
            .filter(F.col("tile_x").between(west, east) & F.col("tile_y").between(south, north)))



def read_bronze_inputs(manifest, work, selected_period=None):
    mobile, coverage, district, worldpop = validate_inventory(manifest["objects"])
    if selected_period:
        mobile = [item for item in mobile if PurePosixPath(item["key"]).name.startswith(selected_period)]
        if len(mobile) != 1:
            raise ValueError(f"Bronze no contiene el trimestre solicitado: {selected_period}")
    paths = [verified_uri(item) for item in mobile]
    periods = sorted(object_period(item) for item in mobile)
    return paths, periods, verified_uri(coverage), load_districts(downloaded_object(district, work)), downloaded_object(worldpop, work)
print("Preparando los originales verificados del lote ...", flush=True)
temporary = Path(resources.enter_context(TemporaryDirectory(prefix="benchmark-", dir=TEMP_DIR)))
inputs = read_bronze_inputs(bronze, temporary, BENCHMARK_PERIOD)
show_values({"Parquet móviles del ensayo": len(inputs[0]), "Distritos": len(inputs[3]),
             "Trimestres": len(inputs[1]), "Factores": ", ".join(f"{x}×" for x in sorted(set(SCALE_FACTORS)))})

Preparando los originales verificados del lote ...


,Indicador,Valor
0,Parquet móviles del ensayo,29
1,Distritos,1890
2,Trimestres,29
3,Factores,"1×, 10×, 100×"


## 6. Controles Silver del lote

In [6]:
def clean_tiles(raw, geo, silver_root, ops_root):
    keys = ["period_id", "quadkey"]
    if "replica_id" in raw.columns:
        keys.append("replica_id")
    inspected = raw.withColumn(
        "motivo",
        F.when(F.col("period_id").isNull(), "periodo_invalido")
        .when(F.col("tests").isNull() | (F.col("tests") <= 0) | F.col("devices").isNull()
              | (F.col("devices") <= 0) | (F.col("devices") > F.col("tests")), "pruebas_o_dispositivos_invalidos")
        .when(F.col("avg_d_kbps").isNull() | (F.col("avg_d_kbps") <= 0), "descarga_invalida")
        .when(F.count("*").over(Window.partitionBy(*keys)) > 1, "clave_duplicada"),
    ).persist(StorageLevel.DISK_ONLY)
    assigned = assign_districts(inspected.filter("motivo IS NULL").drop("motivo"), geo).withColumn(
        "motivo",
        F.when(F.col("coincidencias") == 0, "fuera_de_poligonos")
        .when(F.col("coincidencias") > 1, "asignacion_ambigua"),
    ).persist(StorageLevel.DISK_ONLY)
    try:
        rules = {row.motivo or "validas": row["count"] for row in inspected.groupBy("motivo").count().collect()}
        spatial = {row.motivo or "asignadas": row["count"] for row in assigned.groupBy("motivo").count().collect()}
        if not spatial.get("asignadas") or sum(spatial.values()) != rules.get("validas"):
            raise ValueError("La asignación espacial no produjo teselas válidas o los conteos no concilian.")
        write_table(inspected.filter("motivo IS NOT NULL"), f"{ops_root}/cuarentena/calidad")
        write_table(assigned.filter("motivo IS NOT NULL"), f"{ops_root}/cuarentena/espacial")
        clean = assigned.filter("motivo IS NULL").drop("motivo", "coincidencias")
        uri = write_table(clean, f"{silver_root}/ookla_teselas", ["period_id"])
        completeness = inspected.agg(*[F.count(c).alias(c) for c in
                                    ("period_id", "quadkey", "avg_d_kbps", "tests", "devices")]).first().asDict()
        return uri, {"candidatas": sum(rules.values()), "reglas": rules, "espacial": spatial,
                     "completitud_candidatas": completeness}
    finally:
        assigned.unpersist()
        inspected.unpersist()


def clean_coverage(spark, csv_uri, silver_root):
    raw = (spark.read.option("header", "true").option("sep", ";").option("encoding", "windows-1252")
           .option("mode", "FAILFAST").csv(csv_uri))
    coverage = raw.select(
        F.when(F.trim("UBIGEO_DISTRITO").rlike(r"^\d{1,6}$"),
               F.lpad(F.trim("UBIGEO_DISTRITO"), 6, "0")).alias("ubigeo"),
        F.when(F.trim("UBIGEO_CCPP").rlike(r"^\d{1,10}$"),
               F.lpad(F.trim("UBIGEO_CCPP"), 10, "0")).alias("centro_poblado"),
        F.trim("EMPRESA_OPERADORA").alias("operador"),
        F.trim("PERIODO").alias("periodo_referencia"),
        F.expr("try_to_timestamp(FECHA_CORTE, 'yyyyMMdd')").cast("date").alias("fecha_corte_fuente"),
        F.when(F.trim("4G").isin("0", "1"), F.trim("4G").cast("int")).alias("declara_4g"),
    )
    uri = write_table(coverage, f"{silver_root}/osiptel_cobertura")
    stored = spark.read.parquet(uri)
    counts = stored.agg(
        F.count("*").alias("filas"), F.count("ubigeo").alias("con_ubigeo"),
        F.count("declara_4g").alias("con_dato_4g"),
    ).first().asDict()
    dates = stored.select("periodo_referencia", "fecha_corte_fuente").distinct().take(2)
    if len(dates) != 1 or not dates[0].periodo_referencia or not dates[0].fecha_corte_fuente:
        raise ValueError("OSIPTEL debe contener una sola instantánea con fecha de corte.")
    return uri, counts, dates[0].periodo_referencia, str(dates[0].fecha_corte_fuente)


def silver_tiles(spark, paths, geography, silver_root, ops_root, scale=1):
    geo = spark.sparkContext.broadcast((geography.ubigeo.to_numpy(), geography.geometry.to_wkb().tolist()))
    try:
        raw = read_mobile(spark, paths, geography.total_bounds)
        if scale > 1:
            raw = raw.withColumn("replica_id", F.explode(F.sequence(F.lit(1), F.lit(scale))))
        tiles_uri, quality = clean_tiles(raw, geo, silver_root, ops_root)
    finally:
        geo.destroy()
    return tiles_uri, quality


def silver_context_tables(spark, period_ids, coverage, geography, raster, silver_root):
    districts = district_frame(spark, geography)
    periods = spark.createDataFrame([(period,) for period in period_ids], "period_id int")
    coverage_uri, coverage_quality, reference, cutoff = clean_coverage(spark, coverage, silver_root)
    tables = {
        "distritos": write_table(districts, f"{silver_root}/distritos"),
        "trimestres": write_table(periods, f"{silver_root}/trimestres"),
        "osiptel_cobertura": coverage_uri,
    }
    return tables, coverage_quality, reference, cutoff


def silver_population(spark, geography, raster, silver_root):
    uri = write_table(population_frame(spark, geography, raster), f"{silver_root}/worldpop_2020")
    stats = spark.read.parquet(uri).agg(
        F.count("*").alias("distritos"), F.count("poblacion_2020").alias("con_poblacion"),
        F.sum("pixeles_validos").alias("pixeles_validos"),
    ).first().asDict()
    if stats["distritos"] != len(geography) or not stats["con_poblacion"]:
        raise ValueError("WorldPop Silver no concilia el universo distrital o no contiene población.")
    return uri, stats


def run_silver(spark, bronze_inputs, silver_root, ops_root, scale=1):
    paths, period_ids, coverage, geography, raster = bronze_inputs
    tiles_uri, quality = silver_tiles(spark, paths, geography, silver_root, ops_root, scale=scale)
    tables, coverage_quality, reference, cutoff = silver_context_tables(
        spark, period_ids, coverage, geography, raster, silver_root
    )
    tables["ookla_teselas"] = tiles_uri
    tables["worldpop_2020"], _ = silver_population(spark, geography, raster, silver_root)
    return {"tables": tables, "quality": quality, "coverage_quality": coverage_quality,
            "osiptel_periodo": reference, "osiptel_fecha_corte": cutoff,
            "distritos": len(geography), "trimestres": len(period_ids)}
show_values({"Fuente de teselas": len(inputs[0]), "Dimensión distrital": len(inputs[3]),
             "Dimensión trimestral": len(inputs[1]), "Cobertura OSIPTEL": inputs[2],
             "Raster WorldPop preparado": inputs[4].name})

,Indicador,Valor
0,Fuente de teselas,29
1,Dimensión distrital,1890
2,Dimensión trimestral,29
3,Cobertura OSIPTEL,s3://american-tower-dev-bronze-b7f9699786f4c119558b9d30f9/run_id=20261125T071603Z-190d23d89e/osiptel_cobertura/cobertura_osiptel.csv
4,Raster WorldPop preparado,per_ppp_2020.tif


## 7. Salidas Gold y evidencias por escenario

In [7]:
def district_panel(spark, tiles, districts, periods, silver, gold_root):
    stats = tiles.groupBy("ubigeo", "period_id").agg(
        (F.sum(F.col("avg_d_kbps").cast("double") * F.col("tests")) /
         F.sum("tests") / 1000).alias("vel_descarga_mbps"),
        F.sum("tests").alias("tests"),
        F.count("*").alias("teselas_observadas"),
    )
    panel = (districts.crossJoin(periods).join(stats, ["ubigeo", "period_id"], "left")
             .withColumn("year", F.floor(F.col("period_id") / 4).cast("int"))
             .withColumn("quarter", (F.col("period_id") % 4 + 1).cast("int"))
             .withColumn("observado", F.col("vel_descarga_mbps").isNotNull()))
    uri = write_table(panel, f"{gold_root}/panel_distrital", ["year"])
    counts = spark.read.parquet(uri).agg(
        F.count("*").alias("filas"),
        F.count("vel_descarga_mbps").alias("observadas"),
        F.count_distinct("ubigeo").alias("distritos"),
        F.count_distinct("period_id").alias("trimestres"),
        F.count_distinct(F.struct("ubigeo", "period_id")).alias("claves"),
        F.sum("teselas_observadas").alias("teselas"),
    ).first().asDict()
    if (counts["distritos"] != silver["distritos"] or counts["trimestres"] != silver["trimestres"]
            or counts["filas"] != counts["distritos"] * counts["trimestres"]
            or counts["filas"] != counts["claves"] or not counts["observadas"]
            or counts["teselas"] != silver["quality"]["espacial"]["asignadas"]):
        raise ValueError("El panel Gold no concilia el grano distrito-trimestre o las teselas publicadas en Silver.")
    return uri, counts


def district_context(spark, districts, population, coverage, silver, gold_root):
    # Una fila por centro poblado. Si al menos una operadora declara 4G, el centro cuenta como positivo.
    towns = (coverage.filter("centro_poblado IS NOT NULL AND ubigeo IS NOT NULL")
             .groupBy("ubigeo", "centro_poblado").agg(F.max("declara_4g").alias("alguna_4g")))
    district_cov = towns.groupBy("ubigeo").agg(
        F.count("*").alias("centros_en_reporte"),
        F.count("alguna_4g").alias("centros_con_dato_4g"),
        F.sum("alguna_4g").alias("centros_con_4g_declarada"),
    ).withColumn("periodo_referencia", F.lit(silver["osiptel_periodo"])).withColumn(
        "fecha_corte_fuente", F.to_date(F.lit(silver["osiptel_fecha_corte"])))
    context = districts.join(population, "ubigeo", "left").join(district_cov, "ubigeo", "left")
    uri = write_table(context, f"{gold_root}/contexto_territorial")
    counts = spark.read.parquet(uri).agg(
        F.count("*").alias("filas"), F.count_distinct("ubigeo").alias("claves"),
        F.count("poblacion_2020").alias("con_poblacion"), F.count("centros_en_reporte").alias("con_osiptel"),
    ).first().asDict()
    if (counts["filas"] != silver["distritos"] or counts["filas"] != counts["claves"]
            or not counts["con_poblacion"] or not counts["con_osiptel"]):
        raise ValueError("El contexto Gold no concilia la dimensión distrital, WorldPop o la cobertura OSIPTEL.")
    return uri, counts


def run_gold(spark, silver, gold_root):
    tables = silver["tables"]
    districts = spark.read.parquet(tables["distritos"])
    panel_uri, panel = district_panel(spark, spark.read.parquet(tables["ookla_teselas"]),
                                      districts, spark.read.parquet(tables["trimestres"]), silver, gold_root)
    context_uri, context = district_context(spark, districts, spark.read.parquet(tables["worldpop_2020"]),
                                            spark.read.parquet(tables["osiptel_cobertura"]), silver, gold_root)
    return {"panel_uri": panel_uri, "panel": panel, "context_uri": context_uri, "context": context}

from py4j.protocol import Py4JError

def stage_metrics(context, group):
    fields = ("inputBytes", "outputBytes", "shuffleReadBytes", "shuffleWriteBytes",
              "memoryBytesSpilled", "diskBytesSpilled", "executorRunTime", "executorCpuTime")
    ids = set()
    try:
        tracker = context.statusTracker()
        for job in tracker.getJobIdsForGroup(group):
            info = tracker.getJobInfo(job)
            if info:
                ids.update(info.stageIds)
        if not context.uiWebUrl or not ids:
            return {"stage_ids": sorted(ids), "resources": None}
        app = quote(context.applicationId, safe="")
        totals = dict.fromkeys(fields, 0)
        for stage in sorted(ids):
            url = f"{context.uiWebUrl}/api/v1/applications/{app}/stages/{stage}?details=false"
            with urlopen(url, timeout=4) as response:
                attempts = json.load(response)
            for attempt in attempts:
                if attempt["status"] == "COMPLETE":
                    for field in fields:
                        totals[field] += attempt[field]
        return {"stage_ids": sorted(ids), "resources": totals}
    except (OSError, ValueError, KeyError, Py4JError) as error:
        return {"stage_ids": sorted(ids), "resources": None, "spark_ui_error": type(error).__name__}



def run_factor(spark, inputs, root, factor, lineage, trials):
    prefix = f"{root}/escala={factor}x"
    group = f"escala-{factor}x"
    context = spark.sparkContext
    before = executor_usage(context)
    context.setJobGroup(group, f"Bronze a Gold {factor}x")
    try:
        start = time.perf_counter()
        silver = run_silver(spark, inputs, f"{prefix}/silver", f"{prefix}/ops", scale=factor)
        silver_done = time.perf_counter()
        gold = run_gold(spark, silver, f"{prefix}/gold")
        gold_done = time.perf_counter()
        after = executor_usage(context)
        quality = silver["quality"]
        candidates, assigned = quality["candidatas"], quality["espacial"]["asignadas"]
        observed = gold["panel"]["observadas"]
        equivalent = [candidates // factor, assigned // factor, observed]
        if candidates % factor or assigned % factor or (trials and equivalent != trials[0]["normalizado"]):
            raise ValueError("La réplica no conserva los conteos esperados ni la cobertura del lote.")
        involved = None
        if before["activos"] is not None and after["activos"] is not None:
            involved = [id_ for id_, tasks in after["tareas"].items() if tasks > before["tareas"].get(id_, 0)]
        hosts = None if involved is None else len({after["hosts"][id_] for id_ in involved if after["hosts"].get(id_)})
        row = {"factor": factor, "seconds": gold_done - start, "silver_seconds": silver_done - start,
               "gold_seconds": gold_done - silver_done, "candidatas": candidates, "asignadas": assigned,
               "gold_observadas": observed, "normalizado": equivalent,
               "executors_activos_antes": before["activos"], "executors_activos_despues": after["activos"],
               "executors_con_tareas": involved, "workers_con_tareas": hosts,
               "application_id": context.applicationId, **stage_metrics(context, group)}
        row["uri_metrica"] = save_json(f"{prefix}/metricas.json", {**lineage, **row})
        return row
    except Exception as error:
        try:
            save_json(f"{prefix}/fallo.json", {**lineage, "factor": factor, "error": type(error).__name__,
                      "application_id": context.applicationId})
        except ClientError as write_error:
            print(f"No se pudo publicar el diagnóstico de {factor}×: {write_error}", flush=True)
        raise
    finally:
        for key in ("spark.jobGroup.id", "spark.job.description", "spark.job.interruptOnCancel"):
            context.setLocalProperty(key, None)


def show_scale(row):
    show_values({"Factor": row["factor"], "Segundos totales": round(row["seconds"], 3),
                 "Segundos Silver": round(row["silver_seconds"], 3), "Segundos Gold": round(row["gold_seconds"], 3),
                 "Teselas candidatas": row["candidatas"], "Teselas asignadas": row["asignadas"],
                 "Distrito-trimestre observados": row["gold_observadas"],
                 "Executors activos antes / después": f"{row['executors_activos_antes']} / {row['executors_activos_despues']}",
                 "Workers con tareas registradas": row["workers_con_tareas"], "Evidencia": row["uri_metrica"]})
    if row["resources"] is not None:
        show_values(row["resources"])
    else:
        print("Métricas de recursos no disponibles en la interfaz Spark para este escenario.")
show_rows([{"factor": factor, "silver": f"{root}/escala={factor}x/silver/",
            "gold": f"{root}/escala={factor}x/gold/", "metrica": f"{root}/escala={factor}x/metricas.json"}
           for factor in sorted(set(SCALE_FACTORS))])

,factor,silver,gold,metrica
0,1,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=1x/silver/,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=1x/gold/,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=1x/metricas.json
1,10,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=10x/silver/,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=10x/gold/,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=10x/metricas.json
2,100,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=100x/silver/,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=100x/gold/,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=100x/metricas.json


## 8. Escenario de escala 1×

In [8]:
trials = []


def execute_scenario(factor):
    if factor not in SCALE_FACTORS:
        print(f"{factor}× omitido por SCALE_FACTORS.")
        return
    print(f"Inicia el escenario {factor}× · {root}", flush=True)
    row = run_factor(spark, inputs, root, factor, lineage, trials)
    trials.append(row)
    show_scale(row)


execute_scenario(1)

Inicia el escenario 1× · s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento


,Indicador,Valor
0,Factor,1
1,Segundos totales,185.241
2,Segundos Silver,172.951
3,Segundos Gold,12.29
4,Teselas candidatas,555697
5,Teselas asignadas,417420
6,Distrito-trimestre observados,41434
7,Executors activos antes / después,8 / 8
8,Workers con tareas registradas,8
9,Evidencia,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=1x/metricas.json


,Indicador,Valor
0,inputBytes,1331799188
1,outputBytes,26371661
2,shuffleReadBytes,29387405
3,shuffleWriteBytes,29387405
4,memoryBytesSpilled,0
5,diskBytesSpilled,0
6,executorRunTime,2644379
7,executorCpuTime,1746913828579


## 9. Escenario de escala 10×

In [9]:
execute_scenario(10)

Inicia el escenario 10× · s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento


,Indicador,Valor
0,Factor,10
1,Segundos totales,236.86
2,Segundos Silver,225.211
3,Segundos Gold,11.649
4,Teselas candidatas,5556970
5,Teselas asignadas,4174200
6,Distrito-trimestre observados,41434
7,Executors activos antes / después,8 / 8
8,Workers con tareas registradas,8
9,Evidencia,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=10x/metricas.json


,Indicador,Valor
0,inputBytes,2370863809
1,outputBytes,139071378
2,shuffleReadBytes,193459161
3,shuffleWriteBytes,193459161
4,memoryBytesSpilled,0
5,diskBytesSpilled,0
6,executorRunTime,3989069
7,executorCpuTime,1803469510863


## 10. Escenario de escala 100×

In [10]:
execute_scenario(100)

Inicia el escenario 100× · s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento


,Indicador,Valor
0,Factor,100
1,Segundos totales,838.078
2,Segundos Silver,828.079
3,Segundos Gold,10.0
4,Teselas candidatas,55569700
5,Teselas asignadas,41742000
6,Distrito-trimestre observados,41434
7,Executors activos antes / después,8 / 8
8,Workers con tareas registradas,8
9,Evidencia,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escala=100x/metricas.json


,Indicador,Valor
0,inputBytes,9700832653
1,outputBytes,866752448
2,shuffleReadBytes,1343491780
3,shuffleWriteBytes,1343491780
4,memoryBytesSpilled,0
5,diskBytesSpilled,0
6,executorRunTime,17583465
7,executorCpuTime,2345901233467


## 11. Comparación final y registro de resultados

In [11]:
base_seconds = next((row["seconds"] for row in trials if row["factor"] == 1), None)
overview = []
for row in trials:
    overview.append({"factor": row["factor"], "segundos": round(row["seconds"], 3),
                     "segundos_vs_1x": round(row["seconds"] / base_seconds, 3) if base_seconds else None,
                     "candidatas": row["candidatas"], "workers_con_tareas": row["workers_con_tareas"]})
show_rows(overview)
worker_counts = {row["workers_con_tareas"] for row in trials if row["workers_con_tareas"] is not None}
if len(worker_counts) > 1:
    print("El número de workers con tareas varió entre escenarios. Considerar esa diferencia al comparar los tiempos.")
execution = {"application_id": spark.sparkContext.applicationId, "spark": spark.version,
             "executors_requested": EXECUTORS, "executor_cores": EXECUTOR_CORES,
             "executor_memory": EXECUTOR_MEMORY, "executor_overhead": EXECUTOR_OVERHEAD,
             "shuffle_partitions": SHUFFLE_PARTITIONS, "spark_event_log_uri": events_uri}
report = {**lineage, **execution, "period_scope": BENCHMARK_PERIOD or "todos_los_trimestres_de_bronze",
          "timing_scope": "Lecturas Spark y transformaciones y escrituras Silver y Gold",
          "union_uri": join_uri, "trials": trials, "comparison": overview}
resultado = {**report, "join_uri": join_uri, "scaling_uri": save_json(f"{root}/escalabilidad.json", report),
             "expected_rows": join["expected_rows"], "median_seconds": join["median_seconds"],
             "reduction_percent": join["reduction_percent"]}
show_values({"Unión y planes": join_uri, "Escalabilidad": resultado["scaling_uri"], "Eventos Spark": events_uri})
resources.close()

,factor,segundos,segundos_vs_1x,candidatas,workers_con_tareas
0,1,185.241,1.000,555697,8
1,10,236.860,1.279,5556970,8
2,100,838.078,4.524,55569700,8


,Indicador,Valor
0,Unión y planes,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/union.json
1,Escalabilidad,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/rendimiento/escalabilidad.json
2,Eventos Spark,s3://american-tower-dev-operations-1dd72256126674a5893dfca3df/run_id=20261125T073537Z-c20fc75eae/eventos_spark/
